# Bank Statement Transaction Extraction with Llama Vision

Specialized notebook for extracting transactional data from bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Handles empty debit/credit cells correctly
- Preserves row-by-row transaction structure
- Calculates running balances
- Validates transaction integrity

# Bank Statement Extraction with Evaluation - Llama Vision

Enhanced notebook for extracting and evaluating bank statement data using Llama-3.2-Vision-Instruct.

## Updates for Evaluation Data Integration

### ✅ New Features:
1. **Multi-Bank Processing**: Process all Big 4 Australian bank statements (CommBank, ANZ, NAB, Westpac)
2. **Ground Truth Integration**: Load and compare with `evaluation_data/ground_truth.csv`
3. **Evaluation Metrics**: 
   - Bank name recognition accuracy
   - Account holder extraction accuracy
   - Transaction count matching
   - Transaction date overlap analysis
4. **Comprehensive Reporting**:
   - Overall performance score
   - Field-level accuracy metrics
   - Per-bank detailed results
   - CSV export of evaluation results

### 📁 Data Sources:
- **Images**: `evaluation_data/` directory containing:
  - `commbank_statement_001.png`
  - `anz_statement_001.png`
  - `nab_statement_001.png`
  - `westpac_statement_001.png`
- **Ground Truth**: `evaluation_data/ground_truth.csv` with expected values

### 🎯 Evaluation Workflow:
1. Load ground truth data for bank statements
2. Process each bank statement image
3. Extract transaction data and account details
4. Compare extracted vs expected values
5. Generate evaluation report with accuracy metrics
6. Export results to CSV for further analysis

In [1]:
# Configuration
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Big 4 Australian Bank Statement Images
BANK_STATEMENT_IMAGES = {
    "CommBank": "evaluation_data/commbank_statement_001.png",
    "ANZ": "evaluation_data/anz_statement_001.png", 
    "NAB": "evaluation_data/nab_statement_001.png",
    "Westpac": "evaluation_data/westpac_statement_001.png"
}

# Single statement for quick testing
STATEMENT_IMAGE_PATH = BANK_STATEMENT_IMAGES["CommBank"]

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# Model loading settings - NO QUANTIZATION BY DEFAULT (H200 has plenty of memory)
USE_QUANTIZATION = False  # H200 doesn't need quantization
LOAD_IN_8BIT = False  # 8-bit causes inference issues with bitsandbytes  
DEVICE_MAP = "auto"  # Automatic device mapping
MAX_NEW_TOKENS = 3000  # Increased for full statement extraction

In [2]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from PIL import Image
import json
import re
from typing import Dict, List, Any, Optional, Tuple
import pandas as pd
from datetime import datetime
from decimal import Decimal, InvalidOperation
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Initialize model and processor
print("Loading Llama Vision model for bank statement processing...")

# Load model WITHOUT quantization by default
# The bitsandbytes 8-bit quantization can cause RuntimeError during inference
if USE_QUANTIZATION and LOAD_IN_8BIT:
    print("⚠️ Loading with 8-bit quantization (may cause inference errors)")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
        load_in_8bit=True,
    )
else:
    print("✅ Loading model without quantization (recommended for H200)")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
    )

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("✅ Model loaded successfully")
print(f"📊 Device: {next(model.parameters()).device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")

Loading Llama Vision model for bank statement processing...
✅ Loading model without quantization (recommended for H200)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Model loaded successfully
📊 Device: cuda:0
🎮 GPU: NVIDIA H200
💾 Memory: 9.76GB allocated


In [4]:
# Specialized bank statement extraction prompts - FOCUSED ON CSV AND MARKDOWN
BANK_STATEMENT_PROMPTS = {
    "csv_extraction": """Extract this bank statement as CSV format with proper structure.

TRANSACTION CLASSIFICATION RULES:
- DEBITS (money OUT): Direct debits, ATM withdrawals, fees, purchases, transfers sent
- CREDITS (money IN): Salary/wages, transfers received, deposits, refunds

CRITICAL REQUIREMENTS:
1. Extract header info first (bank, account holder, dates)
2. Convert ALL transactions to CSV with proper empty cell handling
3. Use DD/MM/YYYY date format
4. Remove $ symbols and CR/DR suffixes
5. Empty debit/credit cells = blank (not "null" or "0")

OUTPUT FORMAT:
**STATEMENT DETAILS:**
- Bank: [Bank Name]
- Account Holder: [Full Name] 
- Account: [Account Number]
- Period: [DD/MM/YYYY to DD/MM/YYYY]
- Opening Balance: [Amount]
- Closing Balance: [Amount]

**TRANSACTIONS CSV:**
```
Date,Description,Debit,Credit,Balance
04/09/2025,Direct Debit DOMINO'S PTY LTD,117.57,,8586.28
01/09/2025,Monthly Service Fee,17.89,,8703.85
31/08/2025,Salary Payment,,2500.00,8721.74
```

Extract ALL visible transactions maintaining exact row structure.""",

    "markdown_table": """Extract this bank statement as a clean markdown table.

CLASSIFICATION LOGIC:
- Service fees, ATM withdrawals, direct debits = DEBIT
- Salary payments, transfers in, deposits = CREDIT

FORMAT REQUIREMENTS:
- Dates: DD/MM/YYYY format
- Amounts: Clean numbers (no $ or CR/DR)
- Empty cells: Leave blank

OUTPUT:
## Bank Statement Extract

**Account Information:**
- **Bank:** Commonwealth Bank of Australia
- **Account Holder:** DAVID ROBERT BROWN  
- **Account Number:** 979918123
- **Statement Period:** 07/08/2025 to 06/09/2025
- **Opening Balance:** 9188.50
- **Closing Balance:** 8586.28

**Transactions:**

| Date | Description | Debit | Credit | Balance |
|------|-------------|--------|--------|---------|
| 04/09/2025 | Direct Debit DOMINO'S PTY LTD | 117.57 |  | 8586.28 |
| 01/09/2025 | Monthly Service Fee | 17.89 |  | 8703.85 |
| 31/08/2025 | Salary Payment |  | 2500.00 | 8721.74 |

Extract all visible transactions in this exact format.""",

    "structured_csv": """Convert this bank statement to structured CSV with header information.

TRANSACTION TYPES:
- OUTGOING (Debit): Fees, withdrawals, payments, purchases
- INCOMING (Credit): Salary, deposits, transfers received

RULES:
1. Clean date format: DD/MM/YYYY
2. No currency symbols or suffixes  
3. Blank cells for empty debit/credit (not zero)
4. Preserve exact transaction order

First provide account details, then CSV table:

ACCOUNT DETAILS:
Bank Name: [Extract from header]
Account Holder: [Extract full name]
Account Number: [Extract number] 
Statement Period: [DD/MM/YYYY to DD/MM/YYYY]

TRANSACTIONS:
Date,Description,Debit,Credit,Balance
[Extract each row maintaining structure]""",

    "row_by_row_extraction": """Extract the transaction table from this bank statement image ROW BY ROW.

CRITICAL: This is a TABLE where each row represents one transaction.

For the transaction table:
1. Identify ALL column headers (typically: Date, Description, Debit, Credit, Balance)
2. For EACH row in the table, extract ALL cells
3. If a cell is EMPTY (like debit column for a credit transaction), mark it as "EMPTY"
4. Preserve EXACT row order - do not skip any rows

Format your response EXACTLY as:
COLUMNS: Date | Description | Debit | Credit | Balance
ROW 1: [date] | [description] | [amount or EMPTY] | [amount or EMPTY] | [balance]
ROW 2: [date] | [description] | [amount or EMPTY] | [amount or EMPTY] | [balance]
[continue for all rows]

Remember: Each transaction has EITHER debit OR credit, so one will always be EMPTY.""",

    "csv_with_validation": """Extract bank statement to CSV format with validation checks.

CLASSIFICATION VALIDATION:
- Monthly Service Fee → DEBIT (bank charges you)
- ATM Withdrawal → DEBIT (taking cash out)
- Direct Debit [Company] → DEBIT (payment to company)
- Salary/Wage Payment → CREDIT (money coming in)
- Transfer From [Name] → CREDIT (money received)

FORMAT:
1. Account summary first
2. CSV table with proper empty cell handling
3. Validation notes for any questionable classifications

ACCOUNT INFO:
Bank: [Extract]
Holder: [Extract] 
Number: [Extract]
Period: [Extract]

CSV DATA:
Date,Description,Debit,Credit,Balance
[All transactions with proper classification]

VALIDATION NOTES:
[Flag any uncertain classifications]""",

    "enhanced_markdown": """Extract this bank statement as a comprehensive markdown report.

OUTPUT STRUCTURE:

# Bank Statement Analysis

## Account Summary
- **Institution:** [Bank Name from header]
- **Account Holder:** [Full name from statement]
- **Account Number:** [Number as shown]
- **Statement Period:** [DD/MM/YYYY to DD/MM/YYYY]
- **Opening Balance:** [Clean amount]
- **Closing Balance:** [Clean amount]

## Transaction Details

| Date | Description | Type | Amount | Balance |
|------|-------------|------|--------|---------|
| DD/MM/YYYY | [Full description] | Debit/Credit | [Amount] | [Balance] |

## Transaction Summary
- **Total Debits:** [Sum of outgoing]
- **Total Credits:** [Sum of incoming] 
- **Net Change:** [Difference]
- **Transaction Count:** [Number of rows]

Use this exact format with clean formatting (no $, no CR/DR suffixes)."""
}

In [5]:
# Enhanced prompts for complex bank statements with multiple sections
COMPLEX_STATEMENT_PROMPTS = {
    "full_statement_extraction": """Analyze this complete bank statement image that contains multiple sections.

DOCUMENT STRUCTURE:
1. HEADER/ACCOUNT SUMMARY (top section)
2. TRANSACTION TABLE (main middle section)  
3. SUMMARY/TOTALS (bottom section)

Extract the following:

SECTION 1 - ACCOUNT INFORMATION (from top of statement):
- Bank name and logo area
- Account holder name
- Account number (may be partially masked)
- Statement period (e.g., "01 Mar 2024 to 31 Mar 2024")
- Opening balance
- Total deposits/credits for period
- Total withdrawals/debits for period
- Closing balance

SECTION 2 - TRANSACTION TABLE:
Focus on the main table with columns. For EACH transaction row:
- Date (transaction date)
- Description/Particulars (transaction details)
- Debit/Withdrawal (amount if debit, else null)
- Credit/Deposit (amount if credit, else null)
- Balance (running balance after transaction)

CRITICAL RULES FOR THE TABLE:
- Each row is ONE transaction
- A transaction has EITHER debit OR credit, never both
- Empty cells must be marked as null
- Preserve exact row order

SECTION 3 - SUMMARY (if present at bottom):
- Total number of transactions
- Sum of all debits
- Sum of all credits
- Final closing balance
- Any notes or messages

Return as structured JSON with all three sections clearly separated.""",

    "table_focused_with_context": """Extract the transaction table from this bank statement, understanding it exists within a larger document.

IMPORTANT: This statement has multiple sections, but focus primarily on the TRANSACTION TABLE in the middle.

Before the table, note:
- Statement date range (if visible)
- Opening balance (if shown above table)

For the TRANSACTION TABLE:
1. Identify exact column headers (Date, Description, Debit, Credit, Balance, etc.)
2. Extract EVERY row maintaining structure
3. Empty cells (like debit for a deposit) = "EMPTY"
4. Do not confuse summary rows with transactions

After the table, note:
- Any summary totals shown
- Closing balance

Format as:
STATEMENT PERIOD: [dates]
OPENING BALANCE: [amount]

TABLE COLUMNS: Date | Description | Debit | Credit | Balance
ROW 1: [values with EMPTY for blank cells]
ROW 2: [values with EMPTY for blank cells]
[all rows]

SUMMARY:
Total Debits: [amount]
Total Credits: [amount]
Closing Balance: [amount]""",

    "hierarchical_extraction": """Process this multi-section bank statement hierarchically.

STEP 1 - Identify document sections:
- Where does the header/account info end?
- Where does the transaction table begin and end?
- Is there a summary section at the bottom?

STEP 2 - Extract header information:
{
  "header": {
    "bank": "...",
    "account_holder": "...",
    "account_number": "...",
    "period": "...",
    "opening_balance": "..."
  }
}

STEP 3 - Extract transaction table:
{
  "transactions": [
    {
      "row_number": 1,
      "date": "...",
      "description": "...",
      "debit": null or amount,
      "credit": null or amount,
      "balance": "..."
    }
  ]
}

STEP 4 - Extract summary if present:
{
  "summary": {
    "total_transactions": ...,
    "total_debits": "...",
    "total_credits": "...",
    "closing_balance": "..."
  }
}

Return complete JSON combining all sections."""
}

In [6]:
def extract_bank_statement(
    image_path: str, 
    prompt_type: str = "csv_extraction",
    custom_prompt: str = None,
    temperature: float = 0.1
) -> Dict[str, Any]:
    """
    Extract bank statement data from an image.
    
    Args:
        image_path: Path to bank statement image
        prompt_type: Type of extraction prompt
        custom_prompt: Optional custom prompt
        temperature: Generation temperature (lower = more deterministic)
    
    Returns:
        Dictionary with extraction results
    """
    # Load image
    image = Image.open(image_path)
    
    # Select prompt - default to csv_extraction if not found
    prompt = custom_prompt if custom_prompt else BANK_STATEMENT_PROMPTS.get(
        prompt_type, BANK_STATEMENT_PROMPTS["csv_extraction"]
    )
    
    # Prepare inputs
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Apply chat template
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Process inputs
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate response
    print("🔍 Analyzing bank statement...")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=temperature,
            top_p=0.95
        )
    
    # Decode response
    response = processor.decode(output[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    return {
        "raw_response": response,
        "prompt_type": prompt_type,
        "image_path": image_path,
        "extraction_time": datetime.now().isoformat()
    }

In [7]:
def parse_statement_json(response: str) -> Optional[Dict]:
    """
    Parse JSON response from bank statement extraction.
    """
    try:
        # Find JSON in response
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            return data
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
    return None

def parse_row_format(response: str) -> pd.DataFrame:
    """
    Parse row-by-row format response into DataFrame.
    """
    lines = response.strip().split('\n')
    
    # Find column headers
    columns = None
    rows = []
    
    for line in lines:
        if line.startswith('COLUMNS:'):
            columns = [col.strip() for col in line.replace('COLUMNS:', '').split('|')]
        elif line.startswith('ROW'):
            # Extract row data
            row_data = line.split(':', 1)[1] if ':' in line else line
            values = [val.strip() for val in row_data.split('|')]
            
            # Replace EMPTY with None
            values = [None if v.upper() == 'EMPTY' else v for v in values]
            rows.append(values)
    
    if columns and rows:
        return pd.DataFrame(rows, columns=columns)
    return None

def parse_csv_format(response: str) -> pd.DataFrame:
    """
    Parse CSV format with proper handling of empty cells.
    """
    try:
        lines = response.strip().split('\n')
        csv_lines = []
        
        for line in lines:
            # Only include lines that look like CSV
            if ',' in line and not line.startswith('#'):
                csv_lines.append(line)
        
        if csv_lines:
            from io import StringIO
            csv_text = '\n'.join(csv_lines)
            df = pd.read_csv(StringIO(csv_text))
            
            # Replace NaN with None for clarity
            df = df.where(pd.notnull(df), None)
            return df
    except Exception as e:
        print(f"❌ CSV parsing error: {e}")
    return None

In [8]:
def validate_transactions(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Validate bank statement transactions.
    
    Returns:
        Dictionary with validation results and issues
    """
    issues = []
    
    # Check for required columns
    required_cols = ['date', 'description', 'debit', 'credit', 'balance']
    df_cols_lower = [col.lower() for col in df.columns]
    
    for req_col in required_cols:
        if req_col not in df_cols_lower:
            issues.append(f"Missing required column: {req_col}")
    
    # Normalize column names
    df.columns = [col.lower() for col in df.columns]
    
    # Validation checks
    for idx, row in df.iterrows():
        # Check debit/credit exclusivity
        has_debit = row.get('debit') not in [None, '', 'nil', 'empty', 'EMPTY']
        has_credit = row.get('credit') not in [None, '', 'nil', 'empty', 'EMPTY']
        
        if has_debit and has_credit:
            issues.append(f"Row {idx}: Has both debit and credit values")
        elif not has_debit and not has_credit:
            # Check if it's not an opening/closing balance row
            desc = str(row.get('description', '')).lower()
            if 'opening' not in desc and 'closing' not in desc:
                issues.append(f"Row {idx}: Missing both debit and credit values")
    
    # Calculate totals
    total_debits = 0
    total_credits = 0
    
    for _, row in df.iterrows():
        try:
            debit_val = row.get('debit')
            if debit_val and str(debit_val).replace('.', '').replace(',', '').isdigit():
                total_debits += float(str(debit_val).replace(',', ''))
            
            credit_val = row.get('credit')
            if credit_val and str(credit_val).replace('.', '').replace(',', '').isdigit():
                total_credits += float(str(credit_val).replace(',', ''))
        except:
            pass
    
    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "total_debits": total_debits,
        "total_credits": total_credits,
        "net_change": total_credits - total_debits,
        "transaction_count": len(df)
    }

In [9]:
def calculate_running_balance(df: pd.DataFrame, opening_balance: float = None) -> pd.DataFrame:
    """
    Calculate or verify running balance for transactions.
    """
    df = df.copy()
    
    # Normalize column names
    df.columns = [col.lower() for col in df.columns]
    
    # Convert amounts to float
    def to_float(val):
        if val in [None, '', 'nil', 'empty', 'EMPTY', 'null']:
            return 0.0
        try:
            return float(str(val).replace(',', '').replace('$', ''))
        except:
            return 0.0
    
    df['debit_amount'] = df['debit'].apply(to_float)
    df['credit_amount'] = df['credit'].apply(to_float)
    df['balance_amount'] = df['balance'].apply(to_float)
    
    # Calculate expected balance
    if opening_balance is None and len(df) > 0:
        # Try to infer opening balance from first row
        first_balance = df.iloc[0]['balance_amount']
        first_debit = df.iloc[0]['debit_amount']
        first_credit = df.iloc[0]['credit_amount']
        opening_balance = first_balance + first_debit - first_credit
    
    running_balance = opening_balance if opening_balance else 0
    df['calculated_balance'] = 0.0
    df['balance_match'] = True
    
    for idx, row in df.iterrows():
        running_balance = running_balance - row['debit_amount'] + row['credit_amount']
        df.at[idx, 'calculated_balance'] = running_balance
        
        # Check if balances match (within tolerance for rounding)
        if row['balance_amount'] > 0:
            diff = abs(running_balance - row['balance_amount'])
            df.at[idx, 'balance_match'] = diff < 0.01
    
    return df

In [10]:
# Display the CSV extraction prompt being used
print("📋 CSV EXTRACTION PROMPT BEING USED:")
print("="*80)
print(BANK_STATEMENT_PROMPTS["csv_extraction"])
print("="*80)

📋 CSV EXTRACTION PROMPT BEING USED:
Extract this bank statement as CSV format with proper structure.

TRANSACTION CLASSIFICATION RULES:
- DEBITS (money OUT): Direct debits, ATM withdrawals, fees, purchases, transfers sent
- CREDITS (money IN): Salary/wages, transfers received, deposits, refunds

CRITICAL REQUIREMENTS:
1. Extract header info first (bank, account holder, dates)
2. Convert ALL transactions to CSV with proper empty cell handling
3. Use DD/MM/YYYY date format
4. Remove $ symbols and CR/DR suffixes
5. Empty debit/credit cells = blank (not "null" or "0")

OUTPUT FORMAT:
**STATEMENT DETAILS:**
- Bank: [Bank Name]
- Account Holder: [Full Name] 
- Account: [Account Number]
- Period: [DD/MM/YYYY to DD/MM/YYYY]
- Opening Balance: [Amount]
- Closing Balance: [Amount]

**TRANSACTIONS CSV:**
```
Date,Description,Debit,Credit,Balance
04/09/2025,Direct Debit DOMINO'S PTY LTD,117.57,,8586.28
01/09/2025,Monthly Service Fee,17.89,,8703.85
31/08/2025,Salary Payment,,2500.00,8721.74
```

Ex

In [11]:
# Extract using CSV format - much better for practical use
print("🔍 EXTRACTING COMMBANK STATEMENT AS CSV...")
print("📄 Image: evaluation_data/commbank_statement_001.png")
print("="*60)

csv_result = extract_bank_statement(
    "evaluation_data/commbank_statement_001.png", 
    prompt_type="csv_extraction"
)

print("\n📊 CSV EXTRACTION OUTPUT:")
print("-"*60)
print(csv_result["raw_response"])
print("-"*60)

🔍 EXTRACTING COMMBANK STATEMENT AS CSV...
📄 Image: evaluation_data/commbank_statement_001.png
🔍 Analyzing bank statement...

📊 CSV EXTRACTION OUTPUT:
------------------------------------------------------------
Here is the extracted bank statement in CSV format with proper structure:

**STATEMENT DETAILS:**
- Bank: Commonwealth Bank of Australia
- Account Holder: DAVID ROBERT BROWN
- Account: BSB: 062-274 Account Number: 979918123
- Period: 07 Aug 2025 to 06 Sep 2025
- Opening Balance: $9188.50
- Closing Balance: $8586.28

**TRANSACTIONS CSV:**
```
Date,Description,Debit,Credit,Balance
04/09/2025,Direct Debit DOMINO'S PTY LTD,117.57,,8586.28
01/09/2025,Monthly Service Fee,17.89,,8703.85
31/08/2025,Monthly Service Fee,18.87,,8721.74
29/08/2025,Wal ATM WBC WESTPAC GLEN WAVE,241.14,,8740.61
27/08/2025,Direct Debit 94924P40133289 MHF 75969,251.33,,9080.28
23/08/2025,Cash Withdrawal ATM SYDNEY NSW,98.53,,8981.75
21/08/2025,Home Loan Payment LN REPAY 56052P41700670,1918.75,,9331.81
21/08/202

In [12]:
# Test MARKDOWN table format - clean human-readable output
print("📝 TESTING MARKDOWN TABLE FORMAT...")
print("📄 Image: evaluation_data/commbank_statement_001.png")
print("="*60)

markdown_result = extract_bank_statement(
    "evaluation_data/commbank_statement_001.png", 
    prompt_type="markdown_table"
)

print("\n📊 MARKDOWN TABLE OUTPUT:")
print("-"*60)
print(markdown_result["raw_response"])
print("-"*60)

📝 TESTING MARKDOWN TABLE FORMAT...
📄 Image: evaluation_data/commbank_statement_001.png
🔍 Analyzing bank statement...

📊 MARKDOWN TABLE OUTPUT:
------------------------------------------------------------
Here is the extracted bank statement in a markdown table format:

**Account Information:**

* **Bank:** Commonwealth Bank of Australia
* **Account Holder:** DAVID ROBERT BROWN
* **Account Number:** 979918123
* **Statement Period:** 07/08/2025 to 06/09/2025
* **Opening Balance:** 9188.50
* **Closing Balance:** 8586.28

**Transactions:**

| Date | Description | Debit | Credit | Balance |
|------|-------------|--------|--------|---------|
| 04/09/2025 | Direct Debit DOMINO'S PTY LTD | 117.57 |  | 8586.28 |
| 01/09/2025 | Monthly Service Fee | 17.89 |  | 8703.85 |
| 31/08/2025 | Monthly Service Fee | 18.87 |  | 8721.74 |
| 29/08/2025 | Wdi ATM WBC WESTPAC GLEN WAVE | 241.14 |  | 8740.61 |
| 27/08/2025 | Direct Debit 94924P40133289 MHF 75969 | 251.33 |  | 9080.28 |
| 23/08/2025 | Cash Withd

In [13]:
# Add CSV parsing and validation functions for the new output format
def parse_csv_from_response(response: str) -> pd.DataFrame:
    """
    Parse CSV data from the structured response.
    """
    try:
        # Find the CSV section in the response
        lines = response.split('\n')
        csv_lines = []
        in_csv = False
        
        for line in lines:
            # Look for CSV header or data
            if line.startswith('Date,') or 'Date,Description,Debit,Credit,Balance' in line:
                in_csv = True
                csv_lines.append(line)
            elif in_csv and ',' in line and not line.startswith('#') and not line.startswith('*'):
                csv_lines.append(line)
            elif in_csv and line.strip() == '':
                break  # End of CSV section
        
        if csv_lines:
            from io import StringIO
            csv_text = '\n'.join(csv_lines)
            df = pd.read_csv(StringIO(csv_text))
            return df
    except Exception as e:
        print(f"❌ CSV parsing error: {e}")
    return None

def extract_account_info(response: str) -> dict:
    """
    Extract account information from the response.
    """
    info = {}
    lines = response.split('\n')
    
    for line in lines:
        line = line.strip()
        if line.startswith('- Bank:') or line.startswith('Bank:'):
            info['bank'] = line.split(':', 1)[1].strip()
        elif line.startswith('- Account Holder:') or line.startswith('Holder:'):
            info['account_holder'] = line.split(':', 1)[1].strip()
        elif line.startswith('- Account:') or line.startswith('Number:'):
            info['account_number'] = line.split(':', 1)[1].strip()
        elif line.startswith('- Period:') or line.startswith('Period:'):
            info['statement_period'] = line.split(':', 1)[1].strip()
        elif line.startswith('- Opening Balance:'):
            info['opening_balance'] = line.split(':', 1)[1].strip()
        elif line.startswith('- Closing Balance:'):
            info['closing_balance'] = line.split(':', 1)[1].strip()
    
    return info

def analyze_transaction_classification(df: pd.DataFrame) -> dict:
    """
    Analyze the debit/credit classification in the CSV data.
    """
    if df is None or df.empty:
        return {"error": "No data to analyze"}
    
    # Count debits and credits
    debit_count = df['Debit'].notna().sum() if 'Debit' in df.columns else 0
    credit_count = df['Credit'].notna().sum() if 'Credit' in df.columns else 0
    total_transactions = len(df)
    
    # Check for classification issues
    issues = []
    for idx, row in df.iterrows():
        desc = str(row.get('Description', '')).lower()
        has_debit = pd.notna(row.get('Debit'))
        has_credit = pd.notna(row.get('Credit'))
        
        # Check for obvious misclassifications
        if 'service fee' in desc and not has_debit:
            issues.append(f"Row {idx+1}: Service fee should be DEBIT")
        elif 'salary' in desc and not has_credit:
            issues.append(f"Row {idx+1}: Salary should be CREDIT")
        elif 'atm withdrawal' in desc and not has_debit:
            issues.append(f"Row {idx+1}: ATM withdrawal should be DEBIT")
        elif has_debit and has_credit:
            issues.append(f"Row {idx+1}: Has both debit AND credit - invalid")
        elif not has_debit and not has_credit:
            issues.append(f"Row {idx+1}: Missing both debit AND credit")
    
    return {
        "total_transactions": total_transactions,
        "debit_count": debit_count,
        "credit_count": credit_count,
        "classification_issues": issues,
        "debit_credit_ratio": f"{debit_count}:{credit_count}"
    }

print("✅ Added CSV parsing and validation functions for structured output")

✅ Added CSV parsing and validation functions for structured output


In [14]:
# # Try the enhanced extraction for complex statements with multiple sections
# print("📄 Extracting COMPLETE bank statement with all sections...")
# print("=" * 60)

# # Use the enhanced prompt for complex statements
# complex_result = extract_bank_statement(
#     STATEMENT_IMAGE_PATH,
#     prompt_type="full_statement_extraction",
#     custom_prompt=COMPLEX_STATEMENT_PROMPTS["full_statement_extraction"]
# )

# print("\n📊 Full Statement Extraction Response (first 2000 chars):")
# print("-" * 60)
# print(complex_result["raw_response"][:2000])
# print("..." if len(complex_result["raw_response"]) > 2000 else "")
# print("-" * 60)

In [15]:
# # Parse the complex statement response
# complex_data = parse_statement_json(complex_result["raw_response"])

# if complex_data:
#     print("✅ Successfully parsed complex bank statement\n")
    
#     # Display all sections
#     if "header" in complex_data:
#         print("🏦 HEADER/ACCOUNT INFORMATION:")
#         for key, value in complex_data["header"].items():
#             print(f"  {key.replace('_', ' ').title()}: {value}")
    
#     if "account_details" in complex_data:
#         print("\n📋 ACCOUNT SUMMARY:")
#         for key, value in complex_data["account_details"].items():
#             print(f"  {key.replace('_', ' ').title()}: {value}")
    
#     if "transactions" in complex_data:
#         trans_df = pd.DataFrame(complex_data["transactions"])
#         print(f"\n📊 TRANSACTION TABLE: {len(trans_df)} transactions")
#         print("\nFirst 10 transactions:")
#         display(trans_df.head(10))
        
#         # Check for empty debit/credit handling
#         has_nulls = trans_df[['debit', 'credit']].isnull().any().any()
#         print(f"\n✅ Empty cell handling: {'Correct (nulls present)' if has_nulls else 'Check - no nulls detected'}")
    
#     if "summary" in complex_data:
#         print("\n📈 SUMMARY SECTION:")
#         for key, value in complex_data["summary"].items():
#             print(f"  {key.replace('_', ' ').title()}: {value}")
# else:
#     print("⚠️ Parsing as JSON failed, trying structured text extraction...")

In [16]:
# # Extract using structured JSON format
# print("📄 Extracting bank statement transactions...")
# print("=" * 60)

# result = extract_bank_statement(
#     STATEMENT_IMAGE_PATH, 
#     prompt_type="structured_transactions"
# )

# print("\n📊 Raw Response (first 1000 chars):")
# print("-" * 60)
# print(result["raw_response"][:1000])
# print("..." if len(result["raw_response"]) > 1000 else "")
# print("-" * 60)

In [17]:
# # Parse and display structured data
# parsed_data = parse_statement_json(result["raw_response"])

# if parsed_data:
#     print("✅ Successfully parsed bank statement data\n")
    
#     # Display account details
#     if "account_details" in parsed_data:
#         print("🏦 Account Details:")
#         for key, value in parsed_data["account_details"].items():
#             print(f"  {key.replace('_', ' ').title()}: {value}")
    
#     # Convert transactions to DataFrame
#     if "transactions" in parsed_data:
#         transactions_df = pd.DataFrame(parsed_data["transactions"])
        
#         print(f"\n📊 Transactions: {len(transactions_df)} found")
#         print("\nFirst 5 transactions:")
#         display(transactions_df.head())
        
#         # Save to CSV
#         output_file = "extracted_bank_transactions.csv"
#         transactions_df.to_csv(output_file, index=False)
#         print(f"\n💾 Saved to {output_file}")
# else:
#     print("⚠️ Could not parse JSON. Trying row-by-row format...")

In [18]:
# # Try row-by-row extraction for better structure preservation
# print("\n🔍 Trying row-by-row extraction...")
# row_result = extract_bank_statement(
#     STATEMENT_IMAGE_PATH,
#     prompt_type="row_by_row_extraction"
# )

# print("\n📊 Row Format Response (first 1500 chars):")
# print("-" * 60)
# print(row_result["raw_response"][:1500])
# print("-" * 60)

# # Parse row format
# row_df = parse_row_format(row_result["raw_response"])
# if row_df is not None:
#     print(f"\n✅ Extracted {len(row_df)} transaction rows")
#     print("\nTransaction Table:")
#     display(row_df)
    
#     # Use this DataFrame for further processing
#     transactions_df = row_df

In [19]:
# # Validate transactions
# if 'transactions_df' in locals() and transactions_df is not None:
#     print("\n🔍 Validating Transactions...")
#     print("=" * 60)
    
#     validation_results = validate_transactions(transactions_df)
    
#     if validation_results["valid"]:
#         print("✅ All transactions passed validation!")
#     else:
#         print("⚠️ Validation issues found:")
#         for issue in validation_results["issues"]:
#             print(f"  - {issue}")
    
#     print(f"\n📊 Transaction Summary:")
#     print(f"  Total Transactions: {validation_results['transaction_count']}")
#     print(f"  Total Debits: ${validation_results['total_debits']:,.2f}")
#     print(f"  Total Credits: ${validation_results['total_credits']:,.2f}")
#     print(f"  Net Change: ${validation_results['net_change']:,.2f}")

In [20]:
# # Calculate and verify running balance
# if 'transactions_df' in locals() and transactions_df is not None:
#     print("\n💰 Calculating Running Balance...")
#     print("=" * 60)
    
#     # Calculate with balance verification
#     balanced_df = calculate_running_balance(transactions_df)
    
#     # Show results
#     print("\nTransactions with Balance Verification:")
#     display_cols = ['date', 'description', 'debit', 'credit', 'balance', 'calculated_balance', 'balance_match']
#     display_cols = [col for col in display_cols if col in balanced_df.columns]
#     display(balanced_df[display_cols].head(10))
    
#     # Check for mismatches
#     mismatches = balanced_df[balanced_df['balance_match'] == False]
#     if len(mismatches) > 0:
#         print(f"\n⚠️ Found {len(mismatches)} balance mismatches:")
#         display(mismatches[display_cols])
#     else:
#         print("\n✅ All balances match calculated values!")
    
#     # Save enhanced version
#     balanced_df.to_csv("bank_transactions_with_validation.csv", index=False)
#     print("\n💾 Saved validated transactions to bank_transactions_with_validation.csv")

In [21]:
# # Advanced: Extract with custom prompt for specific bank formats
# custom_bank_prompt = """
# Extract the transaction table from this bank statement.

# CRITICAL: This is a TABULAR structure where:
# - Each row represents exactly ONE transaction
# - Debit column shows withdrawals/payments (empty for deposits)
# - Credit column shows deposits/receipts (empty for withdrawals)
# - One of debit/credit MUST be empty for each transaction

# For each transaction row, extract these columns IN ORDER:
# 1. Date (as shown)
# 2. Transaction details/description
# 3. Reference number (if present, else "N/A")
# 4. Debit amount (if present, else "0.00")
# 5. Credit amount (if present, else "0.00")  
# 6. Balance

# Format as pipe-separated values:
# Date|Description|Reference|Debit|Credit|Balance
# [actual data rows follow]

# Preserve EXACT row structure - do not combine or skip rows.
# """

# print("\n🎯 Using custom extraction prompt...")
# custom_result = extract_bank_statement(
#     STATEMENT_IMAGE_PATH,
#     custom_prompt=custom_bank_prompt
# )

# print("\nCustom extraction result (first 800 chars):")
# print(custom_result["raw_response"][:800])

In [22]:
# # Generate evaluation summary report - COMMENTED OUT FOR SIMPLIFICATION
# # (This was the comprehensive batch processing evaluation that showed poor results)

# # def generate_evaluation_report(results, evaluation):
# #     """Generate a comprehensive evaluation report."""
# #     # [Previous complex evaluation code here]

# # def export_results_to_csv(results, evaluation):
# #     """Export extraction results to CSV matching evaluation format."""
# #     # [Previous CSV export code here]

# # if 'all_results' in locals() and 'evaluation' in locals():
# #     report_summary = generate_evaluation_report(all_results, evaluation)
# #     overall_score = (
# #         report_summary['field_accuracy']['bank_name'] * 0.2 +
# #         report_summary['field_accuracy']['account_holder'] * 0.2 +
# #         report_summary['field_accuracy']['transaction_count'] * 0.3 +
# #         report_summary['field_accuracy']['date_match'] * 0.3
# #     )
# #     print(f"\n🏆 OVERALL PERFORMANCE SCORE: {overall_score*100:.1f}%")

In [23]:
# # Batch processing for all Big 4 bank statements - COMMENTED OUT FOR SIMPLIFICATION
# # (This was the batch processing that ran on H200 and showed poor results)

# # def process_all_bank_statements(evaluate=True):
# #     """Process all Big 4 bank statements and optionally evaluate against ground truth."""
# #     # [Previous batch processing code here]

# # def evaluate_extractions(extraction_results, ground_truth):
# #     """Compare extracted data with ground truth."""
# #     # [Previous evaluation code here]

# # print("🚀 Processing all Big 4 Australian Bank Statements...")
# # all_results, evaluation = process_all_bank_statements(evaluate=True)

In [24]:
# # Batch processing for multiple statement pages - COMMENTED OUT FOR SIMPLIFICATION

# # def process_multi_page_statement(image_paths: List[str]) -> pd.DataFrame:
# #     """Process multiple pages of a bank statement."""
# #     # [Previous multi-page processing code here]

# # Example usage (uncomment and update paths):
# # statement_pages = ["statement_page1.jpg", "statement_page2.jpg"]
# # multi_page_df = process_multi_page_statement(statement_pages)
# # multi_page_df.to_csv("complete_statement.csv", index=False)

In [25]:
# # Export to different formats - COMMENTED OUT FOR SIMPLIFICATION

# # if 'transactions_df' in locals() and transactions_df is not None:
# #     print("\n📤 Exporting Transactions...")
# #     
# #     # 1. Excel format with formatting
# #     with pd.ExcelWriter('bank_transactions.xlsx', engine='openpyxl') as writer:
# #         transactions_df.to_excel(writer, sheet_name='Transactions', index=False)
# #     
# #     # 2. JSON format for APIs
# #     transactions_json = transactions_df.to_json(orient='records', indent=2)
# #     with open('bank_transactions.json', 'w') as f:
# #         f.write(transactions_json)
# #     
# #     # 3. Accounting software format (QIF)
# #     qif_output = "!Type:Bank\n"
# #     # [QIF generation code here]
# #     with open('bank_transactions.qif', 'w') as f:
# #         f.write(qif_output)
# #     
# #     print("\n📊 Export complete!")

In [26]:
# # Clean up GPU memory - COMMENTED OUT FOR SIMPLIFICATION

# # def cleanup_gpu():
# #     """Clean up GPU memory after processing."""
# #     import gc
# #     gc.collect()
# #     if torch.cuda.is_available():
# #         torch.cuda.empty_cache()
# #         print("🧹 GPU memory cleaned")
# #         print(f"   Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
# #         print(f"   Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

# # Uncomment to clean up
# # cleanup_gpu()

# ## Tips for Bank Statement Extraction - COMMENTED OUT FOR SIMPLIFICATION

# ### This section contained best practices and tips for bank statement extraction
# ### Key points were:
# ### - Empty Debit/Credit cell handling 
# ### - Row structure preservation
# ### - Balance verification
# ### - Export format options
# ### - Common bank statement formats

# ### The notebook has been simplified to focus on single CommBank statement extraction
# ### Complex batch processing and evaluation features have been commented out
# ### based on poor results from the H200 run (0% account holder extraction, 
# ### significant transaction count mismatches, 0% date matching)